In [1]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

#

# Ortam, paketler

In [2]:
# Kaggle ortamında genelde bunlar hazır olur ama kontrol etmek zarar vermez.
!pip install -q gensim sentence_transformers --break-system-packages

# (HuggingFace tokenizers uyarı alırsan TOKENIZERS_PARALLELISM env ayarla)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Veri yükleme

In [3]:
import pandas as pd
# NOT: Orijinal Kaggle veri setine bu ortamda internet erişimi yok.
# Bu nedenle notebook'un calisir oldugunu gostermek icin ayni sema (Tweets, Class)
# ile olusturulmus SENTETIK/ORNEK bir veri seti kullanildi.
df = pd.read_csv("/home/claude/kaggle_input/turkey-earthquake-relief-tweets-dataset/dataset.csv")
df.head(10)
# label sütununu integer yap
df['Class'] = df['Class'].astype(int)


# 🟦 BÖLÜM 1 — VERİYİ İNCELEME (EDA)

# Temizleme (tek fonksiyon — Türkçe uyumlu, sayıları korur)

In [4]:
import re
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

stop_words = set(stopwords.words("turkish"))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # combining-dot i problemi ve büyük İ
    text = text.replace("i̇", "i").replace("İ", "i").replace("ı̇", "ı")
    # normalize
    text = unicodedata.normalize("NFKC", text)
    # hanya harf, rakam ve türkçe karakter ile boşluk bırak
    text = re.sub(r"[^\w\sçğıöşü0-9]", " ", text)
    # tokenize
    tokens = word_tokenize(text, language="turkish")
    # stopwords temizle
    tokens = [t for t in tokens if t not in stop_words and len(t) > 0]
    return " ".join(tokens)

# uygula (büyük veri varsa .apply ile ilerleyin)
df["clean_text"] = df["Tweets"].apply(preprocess_text)
df[["Tweets","clean_text"]].head(8)


,Tweets,clean_text
0,Yeni telefonumu aldım kamerası gerçekten çok i...,yeni telefonumu aldım kamerası gerçekten iyi ö...
1,Deprem bölgesine giden itfaiye ekiplerine teşe...,deprem bölgesine giden itfaiye ekiplerine teşe...
2,Hafta sonu ailemle vakit geçirdik çok keyifliydi,hafta sonu ailemle vakit geçirdik keyifliydi
3,Kızılay kan bağışı çağrısında bulundu,kızılay kan bağışı çağrısında bulundu
4,Yeni bir tarif denedim akşam yemeği için #deprem,yeni bir tarif denedim akşam yemeği deprem
5,Kahramanmaraş'ta enkaz altında kalanlar için e...,kahramanmaraş ta enkaz altında kalanlar ekiple...
6,Yeni bir dizi izlemeye başladım çok sürükleyici,yeni bir dizi izlemeye başladım sürükleyici
7,Yeni çıkan film gerçekten harikaydı herkese ta...,yeni çıkan film gerçekten harikaydı herkese ta...


# Baseline: Bag of Words + Logistic Regression (sklearn Pipeline)

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df["clean_text"]
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe_bow = Pipeline([
    ("vect", CountVectorizer(max_features=3000, ngram_range=(1,1))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

pipe_bow.fit(X_train, y_train)
y_pred = pipe_bow.predict(X_test)

print("BoW + LR Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


BoW + LR Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        80
           1       1.00      1.00      1.00        80

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160



# TF-IDF + Logistic Regression (aynı pipeline)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

pipe_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=3000, ngram_range=(1,1))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

pipe_tfidf.fit(X_train, y_train)
y_pred_tfidf = pipe_tfidf.predict(X_test)

print("TF-IDF + LR Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print(classification_report(y_test, y_pred_tfidf))


TF-IDF + LR Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        80
           1       1.00      1.00      1.00        80

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160



# Önemli kelimeler (TF-IDF + LR için)

In [7]:
import numpy as np

vec = pipe_tfidf.named_steps["tfidf"]
clf = pipe_tfidf.named_steps["clf"]
feature_names = vec.get_feature_names_out()

coefs = clf.coef_[0]  # binary için tek vektör
topn = 20
top_pos = np.argsort(coefs)[-topn:][::-1]
top_neg = np.argsort(coefs)[:topn]

print("Top positive words:")
for i in top_pos:
    print(feature_names[i], round(coefs[i], 4))

print("\nTop negative words:")
for i in top_neg:
    print(feature_names[i], round(coefs[i], 4))


Top positive words:
yardım 2.1742
kurtarma 1.4106
geçici 1.3486
deprem 1.3152
bölgesine 1.2834
kızılay 1.2592
bölgeye 1.2493
giden 1.2419
destek 1.2333
çadır 1.1339
isteyenler 1.0983
su 1.0474
kan 0.9339
çağrısında 0.9339
bağışı 0.9339
bulundu 0.9339
ulaştı 0.9247
arama 0.9127
ekipleri 0.9127
psikolojik 0.8962

Top negative words:
yeni -2.3877
bir -1.927
gerçekten -1.8197
hafta -1.7506
bugün -1.5305
sonu -1.3279
başladım -1.3222
aldım -1.2861
güzel -1.2161
akşam -1.1313
gidelim -1.1207
yoğun -1.094
izlemek -1.0604
eğlenceliydi -1.0604
arkadaşlarla -1.0604
buluştuk -1.0604
maçı -1.0604
iyi -1.0005
çıktım -0.9972
ürünü -0.9972


# Word2Vec embedding (Gensim) — cümle embedding: kelime embeddinglerin ortalaması

In [8]:
from gensim.models import Word2Vec
sentences = [s.split() for s in df["clean_text"].tolist()]

# küçük bir model — istersen parametreleri büyüt
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4, seed=42)

# fonksiyon: bir cümlenin vektörü = içerideki kelimelerin ortalaması (oov kelimeler atlanır)
import numpy as np
def sentence_vector_w2v(tokens, model):
    vecs = []
    for t in tokens:
        if t in model.wv:
            vecs.append(model.wv[t])
    if len(vecs)==0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# dataset embedding hâline getir
X_w2v = np.vstack([sentence_vector_w2v(s.split(), w2v_model) for s in df["clean_text"]])

# train/test split ve classifier
from sklearn.ensemble import RandomForestClassifier
X_tr, X_te, y_tr, y_te = train_test_split(X_w2v, df["Class"], test_size=0.2, random_state=42, stratify=df["Class"])

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)
y_pred_w2v = rf.predict(X_te)
print("Word2Vec+RF Accuracy:", accuracy_score(y_te, y_pred_w2v))
print(classification_report(y_te, y_pred_w2v))


Word2Vec+RF Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        80
           1       1.00      1.00      1.00        80

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160



# Transformer / Sentence-Transformer embedding (çok güçlü — HuggingFace)

**Not:** Bu hücre (SBERT / sentence-transformers) internet üzerinden HuggingFace'ten model indirmeyi gerektiriyor. Bu çalıştırma ortamının internet erişimi huggingface.co'ya kapalı olduğu için hata verdi. Kaggle üzerinde (`isInternetEnabled: true` ayarıyla) veya internet erişimi olan herhangi bir ortamda bu hücre sorunsuz çalışır.

In [9]:
from sentence_transformers import SentenceTransformer
model_name = "paraphrase-multilingual-MiniLM-L12-v2"  # Türkçe de iyi çalışır
sbert = SentenceTransformer(model_name)

# embed tüm cümleleri (Kaggle'da zaman alabilir)
embeddings = sbert.encode(df["clean_text"].tolist(), show_progress_bar=True, batch_size=64)

# train/test + classifier (logistic)
X_tr, X_te, y_tr, y_te = train_test_split(embeddings, df["Class"], test_size=0.2, random_state=42, stratify=df["Class"])
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_tr, y_tr)
y_pred_sbert = clf.predict(X_te)
print("SBERT + LR Accuracy:", accuracy_score(y_te, y_pred_sbert))
print(classification_report(y_te, y_pred_sbert))


OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.